In [0]:
# 1. Define Databricks Widgets for Production Parameterization
dbutils.widgets.combobox(
    "catalog", "dbr_dev", ["workspace", "dbr_dev"], "Unity Catalog"
)
dbutils.widgets.combobox(
    "bronze_schema",
    "joshuandegwa_bronze",
    ["joshuandegwa_bronze", "bronze"],
    "Bronze schema",
)
# 2. Get the parameters from the widgets
catalog_name = dbutils.widgets.get("catalog")  # dbr_dev
bronze_schema_name = dbutils.widgets.get("bronze_schema")  # joshuandegwa_bronze
# 3. Create schema if it doesn't exist to prevent SCHEMA_NOT_FOUND
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{bronze_schema_name}")

In [0]:
%sql
-- SHOW CATALOGS;

In [0]:
import random
from pyspark.sql.types import (
    BooleanType,
    DoubleType,
    StringType,
    StructField,
    StructType,
)

# 3. Define Schema & Countries
schema = StructType([
    StructField("icao24", StringType(), True),
    StructField("callsign", StringType(), True),
    StructField("origin_country", StringType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("baro_altitude", DoubleType(), True),
    StructField("velocity", DoubleType(), True),
    StructField("true_track", DoubleType(), True),
    StructField("vertical_rate", DoubleType(), True),
    StructField("on_ground", BooleanType(), True),
])

countries = [
    "Kenya",
    "United Kingdom",
    "Germany",
    "United States",
    "Egypt",
    "Ethiopia",
    "Poland",
    "South Africa",
    "France",
    "Nigeria",
]

# 4. Generate Dirty Test Data
all_records = []
for file_idx in range(500):
  for i in range(10):
    country = random.choice(countries)
    callsign = (
        f"KQA{random.randint(100, 999)}"
        if country == "Kenya"
        else f"FLT{random.randint(1000, 9999)}"
    )

    # Injecting chaos: Null business keys & anomalies
    icao24 = (
        None if (file_idx + i) % 15 == 0 else f"04c{random.randint(100, 999)}"
    )
    altitude = (
        -999.0
        if (file_idx + i) % 20 == 0
        else round(random.uniform(500.0, 12000.0), 2)
    )
    latitude = (
        999.9
        if (file_idx + i) % 25 == 0
        else round(random.uniform(33.9, 41.9), 4)
    )

    record = (
        icao24,
        callsign,
        country,
        round(random.uniform(-4.5, 5.0), 4),
        latitude,
        altitude,
        round(random.uniform(50.0, 250.0), 2),
        round(random.uniform(0.0, 360.0), 2),
        round(random.uniform(-10.0, 10.0), 2),
        False,
    )
    all_records.append(record)

    # Injecting duplicates
    if i == 0:
      all_records.append(record)

df_all = spark.createDataFrame(all_records, schema=schema)

# 5. Build dynamic fully-qualified table name targeting lab4 raw
BRONZE_TABLE = f"{catalog_name}.{bronze_schema_name}.lab4_flights_raw"

(
    df_all.repartition(5)
    .write.format("delta")
    .mode("overwrite")
    .saveAsTable(BRONZE_TABLE)
)

print(
    f"✅ Parameterized generation complete! Written to target table:"
    f" {BRONZE_TABLE}"
)